# San Diego MTS Ridership Analysis

## Analysis Scope

This project analyzes San Diego Metropolitan Transit System (MTS) monthly ridership data for fiscal year 2026, covering July 2025 through June 2026.

The analysis focuses on:

- Month-to-month changes in total ridership
- The relationship between Bus and Trolley ridership changes
- Changes in each mode's share of total monthly ridership
- Annual ridership distribution across Bus, Trolley, and Access

In [1]:
# 1. Setup and Data Loading

import pandas as pd
import numpy as np

df = pd.read_excel("../data/monthly-ridership-julyfy27.xlsx")

display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()
display(df.describe())

,FY-2027,Commuter Bus,Contract Services,Directly Operated,Light Rail,Demand Response,Demand Response Taxi,GRAND TOTAL,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NTD Mode->,CB-PT,MB-PT,MB-DO,LR-DO,DR-PT,DR-TX,NaN,NaN,NaN,NaN,NaN
1,2026-07,12338,1691754,1378400,3973795,24501,8796,7089584,NaN,NaN,NaN,NaN
2,2026-08,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
3,2026-09,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
4,2026-10,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN


Shape: (51, 12)
Columns: ['FY-2027', 'Commuter Bus', 'Contract Services', 'Directly Operated', 'Light Rail', 'Demand Response', 'Demand Response Taxi', 'GRAND TOTAL', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']
<class 'pandas.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   FY-2027               43 non-null     str   
 1   Commuter Bus          32 non-null     object
 2   Contract Services     32 non-null     object
 3   Directly Operated     32 non-null     object
 4   Light Rail            32 non-null     object
 5   Demand Response       19 non-null     object
 6   Demand Response Taxi  32 non-null     object
 7   GRAND TOTAL           41 non-null     object
 8   Unnamed: 8            14 non-null     object
 9   Unnamed: 9            14 non-null     object
 10  Unnamed: 10           14 non-null     object
 11  Unnamed: 11           14 non

,FY-2027,Commuter Bus,Contract Services,Directly Operated,Light Rail,Demand Response,Demand Response Taxi,GRAND TOTAL,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
count,43,32,32,32,32,19,32,41,14,14,14,14
unique,27,30,30,30,30,17,30,18,3,3,3,4
top,TOTAL,CB-PT,MB-PT,MB-DO,LR-DO,DR-PT,DR-TX,0,0,0,0,-1
freq,3,2,2,2,2,2,2,22,11,11,11,11


In [2]:
# 2. Data Cleaning and Validation

# Extract only FY2026 summary table
fy2026_df = df.iloc[35:47, :4].copy()

fy2026_df = fy2026_df.rename(
    columns={
        "FY-2027": "Month",
        "Commuter Bus": "Bus",
        "Contract Services": "Trolley",
        "Directly Operated": "Access"
    }
)

fy2026_df["Month"] = pd.to_datetime(fy2026_df["Month"])

fy2026_df[["Bus", "Trolley", "Access"]] = (
    fy2026_df[["Bus", "Trolley", "Access"]]
    .astype("int64")
)

fy2026_df = fy2026_df.reset_index(drop=True)

# Validation Checks
fy2026_df.info() 
assert fy2026_df.shape[0] == 12
assert fy2026_df.isna().sum().sum() == 0

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Month    12 non-null     datetime64[us]
 1   Bus      12 non-null     int64         
 2   Trolley  12 non-null     int64         
 3   Access   12 non-null     int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 516.0 bytes


In [3]:
# 3. Month-to-Month Ridership Changes

# Total monthly ridership / month-to-month change
fy2026_df["Total"] = fy2026_df[["Bus", "Trolley", "Access"]].sum(axis=1)
fy2026_df["total_pct_change"] = fy2026_df["Total"].pct_change() * 100
   
# Absolute and percent change for each transit mode
fy2026_df["bus_change"] = fy2026_df["Bus"].diff()
fy2026_df["bus_pct_change"] = fy2026_df["Bus"].pct_change() * 100

fy2026_df["trolley_change"] = fy2026_df["Trolley"].diff()
fy2026_df["trolley_pct_change"] = fy2026_df["Trolley"].pct_change() * 100

fy2026_df["access_change"] = fy2026_df["Access"].diff()
fy2026_df["access_pct_change"] = fy2026_df["Access"].pct_change() * 100

fy2026_df.loc[1:, "top_mode"] = (
    fy2026_df.loc[
        1:,
        ["bus_change", "trolley_change", "access_change"]
        ]
        .abs()
        .idxmax(axis=1)
)

fy2026_df["top_mode"] = fy2026_df["top_mode"].map(
    {
        "bus_change": "Bus",
        "trolley_change": "Trolley",
        "access_change": "Access"
    }
)


# Total absolute movement by transit mode
bus_total_movement = fy2026_df["bus_change"].abs().sum()
trolley_total_movement = fy2026_df["trolley_change"].abs().sum()
access_total_movement = fy2026_df["access_change"].abs().sum()

mode_change_summary = pd.Series(
    {
        "Bus": bus_total_movement, 
        "Trolley": trolley_total_movement, 
        "Access": access_total_movement
    }
)


# Results
display(fy2026_df["top_mode"].value_counts())
display(mode_change_summary)

print(f"Greatest Total Movement: {mode_change_summary.idxmax()}")
print(f"Greatest Absolute Movement: {mode_change_summary.max():,.0f}")

display(fy2026_df)

top_mode
Bus        6
Trolley    5
Name: count, dtype: int64

Bus        2418894.0
Trolley    2125536.0
Access       39953.0
dtype: float64

Greatest Total Movement: Bus
Greatest Absolute Movement: 2,418,894


,Month,Bus,Trolley,Access,Total,total_pct_change,bus_change,bus_pct_change,trolley_change,trolley_pct_change,access_change,access_pct_change,top_mode
0,2025-07-01,3010898,3761969,32505,6805372,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-08-01,3130466,3507067,31177,6668710,-2.008149,119568.0,3.971174,-254902.0,-6.775760,-1328.0,-4.085525,Trolley
2,2025-09-01,3280748,3671182,32592,6984522,4.735728,150282.0,4.800627,164115.0,4.679551,1415.0,4.538602,Trolley
3,2025-10-01,3714189,3785364,35164,7534717,7.877347,433441.0,13.211652,114182.0,3.110224,2572.0,7.891507,Bus
4,2025-11-01,3016828,3178935,28547,6224310,-17.391589,-697361.0,-18.775593,-606429.0,-16.020362,-6617.0,-18.817541,Bus
5,2025-12-01,2993782,3196717,30831,6221330,-0.047877,-23046.0,-0.763915,17782.0,0.559370,2284.0,8.000841,Bus
6,2026-01-01,3136133,3231986,29544,6397663,2.834330,142351.0,4.754889,35269.0,1.103288,-1287.0,-4.174370,Bus
7,2026-02-01,3073987,3081798,29700,6185485,-3.316492,-62146.0,-1.981612,-150188.0,-4.646926,156.0,0.528026,Trolley
8,2026-03-01,3468851,3619717,33795,7122363,15.146395,394864.0,12.845337,537919.0,17.454713,4095.0,13.787879,Trolley
9,2026-04-01,3497989,3713897,34424,7246310,1.740251,29138.0,0.839990,94180.0,2.601861,629.0,1.861222,Trolley


In [4]:
# 4. Bus and Trolley Ridership Comparison

# Monthly similarity
fy2026_df["bus_trolley_pct_gap"] = (
        fy2026_df["bus_pct_change"] - fy2026_df["trolley_pct_change"]
).abs()

# Directional similarity
fy2026_df["same_direction"] = pd.NA
fy2026_df.loc[1:, "same_direction"] = (
    np.sign(fy2026_df.loc[1:, "bus_pct_change"]) 
    == np.sign(fy2026_df.loc[1:, "trolley_pct_change"])
)

# Overall relationship
correlation = fy2026_df["bus_pct_change"].corr(
    fy2026_df["trolley_pct_change"]
)

print(f"Correlation Coefficient: {correlation:.3f}")

display(
    fy2026_df[[
        "Month",
        "bus_pct_change",
        "trolley_pct_change",
        "bus_trolley_pct_gap",
        "same_direction"
    ]]
)

Correlation Coefficient: 0.812


,Month,bus_pct_change,trolley_pct_change,bus_trolley_pct_gap,same_direction
0,2025-07-01,NaN,NaN,NaN,<NA>
1,2025-08-01,3.971174,-6.775760,10.746934,False
2,2025-09-01,4.800627,4.679551,0.121076,True
3,2025-10-01,13.211652,3.110224,10.101427,True
4,2025-11-01,-18.775593,-16.020362,2.755231,True
5,2025-12-01,-0.763915,0.559370,1.323285,False
6,2026-01-01,4.754889,1.103288,3.651600,True
7,2026-02-01,-1.981612,-4.646926,2.665314,True
8,2026-03-01,12.845337,17.454713,4.609376,True
9,2026-04-01,0.839990,2.601861,1.761871,True


In [5]:
# 5. Monthly Ridership Share by Mode

mode_share = fy2026_df[["Month"]].copy()

mode_share["bus_share"] = (
    fy2026_df["Bus"] / fy2026_df["Total"] * 100
)

mode_share["trolley_share"] = (
    fy2026_df["Trolley"] / fy2026_df["Total"] * 100
)

mode_share["access_share"] = (
    fy2026_df["Access"] / fy2026_df["Total"] * 100
)

display(mode_share)

,Month,bus_share,trolley_share,access_share
0,2025-07-01,44.242960,55.279403,0.477637
1,2025-08-01,46.942602,52.589886,0.467512
2,2025-09-01,46.971690,52.561679,0.466632
3,2025-10-01,49.294340,50.238967,0.466693
4,2025-11-01,48.468473,51.072890,0.458637
5,2025-12-01,48.121254,51.383177,0.495569
6,2026-01-01,49.019978,50.518228,0.461794
7,2026-02-01,49.696782,49.823062,0.480156
8,2026-03-01,48.703654,50.821855,0.474491
9,2026-04-01,48.272693,51.252251,0.475056


In [6]:
# 6. Annual Ridership Share by Mode

annual_summary = pd.DataFrame(
    {
        "Annual Ridership": [
            fy2026_df["Bus"].sum(),
            fy2026_df["Trolley"].sum(),
            fy2026_df["Access"].sum()
        ]
    },
    index=["Bus", "Trolley", "Access"]
)

annual_total = annual_summary["Annual Ridership"].sum()

annual_summary["Annual Share"] = (
    annual_summary["Annual Ridership"] / annual_total * 100
)

display(
    annual_summary.style.format(
        {
            "Annual Ridership": "{:,.0f}",
            "Annual Share": "{:.2f}%"
        }
    )
)

,Annual Ridership,Annual Share
Bus,"38,868,697",47.82%
Trolley,"42,047,078",51.72%
Access,"373,916",0.46%


# 7. Key Findings

## 1. How does total ridership vary month to month, and which transit modes contribute most to those changes?

FY2026 ridership was mostly stable in most months, with the sharpest decline in November 2025 and the strongest rebound in March 2026. Bus usage contributed slightly more overall to month-to-month ridership movement than Trolley usage, while Access usage contributed very little.

## 2. Did Bus and Trolley ridership move together, or were there months where they diverged noticeably?

Bus and Trolley month-to-month percentage changes had a correlation of approximately 0.812, indicating a strong positive relationship during FY2026. This supports the observation that the two modes generally moved together, although a few months - especially August 2025 - showed noticeable divergence.

## 3. How did each mode's share of total ridership change throughout FY2026?

Mode share changed very little over FY2026. Bus and Trolley consistently accounted for virtually all ridership, with Trolley usage generally holding a modest majority, while Access usage remained below 1% throughout the year.

## 4. How much of total annual ridership came from Bus vs. Trolley vs. Access?

Trolley usage accounted for 51.72% of FY2026 ridership compared with 47.82% for the Bus, a difference of 3.90 percentage points. Trolley ridership was about 8.2% higher than Bus ridership overall. Access accounted for just 0.46% of annual ridership.

In [7]:
# 8. Power BI Export

# Ridership data for Questions 1 - 3
powerbi_df = fy2026_df.merge(mode_share, on="Month")

powerbi_df.to_csv("../data/fy2026_powerbi.csv", index=False)


# Data for Question 4
annual_summary_powerbi = annual_summary.copy().reset_index()
annual_summary_powerbi = annual_summary_powerbi.rename(
    columns={"index": "Mode"}
)

annual_summary_powerbi.to_csv("../data/annual_summary_powerbi.csv", index=False)